In [1]:
import os
import json
import pandas as pd

# =========================
# Config
# =========================
USERS_CSV    = "users_10k.csv"
REVIEWS_CSV  = "reviews_10k.csv"
TIPS_CSV     = "tips_10k.csv"
CHECKINS_CSV = "checkin_10k.csv"
BIZ_CSV      = "business_10k.csv"

OUT_DIR = "export_ui_data"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# Load
# =========================
users    = pd.read_csv(USERS_CSV)
reviews  = pd.read_csv(REVIEWS_CSV)
tips     = pd.read_csv(TIPS_CSV)
checkins = pd.read_csv(CHECKINS_CSV)
biz      = pd.read_csv(BIZ_CSV)

print("✅ Loaded files")
print("users   :", users.shape)
print("reviews :", reviews.shape)
print("tips    :", tips.shape)
print("checkins:", checkins.shape)
print("biz(all):", biz.shape)

# =========================
# STRICT restaurant filter
# =========================
def is_restaurant(categories):
    if not isinstance(categories, str):
        return False
    return "restaurants" in categories.lower()

biz_rest = biz[biz["categories"].apply(is_restaurant)].copy()

print("\n🍽️ Restaurant-only businesses:", biz_rest.shape)

# =========================
# Sanity checks
# =========================
def nunique_safe(df, col):
    return int(df[col].nunique()) if col in df.columns else None

print("\n--- Uniqueness ---")
print("unique users        :", nunique_safe(users, "user_id"))
print("unique restaurants  :", nunique_safe(biz_rest, "business_id"))
print("unique biz in reviews:", nunique_safe(reviews, "business_id"))

# =========================
# Filter events to restaurant-only businesses
# =========================
restaurant_ids = set(biz_rest["business_id"].astype(str))

reviews_r  = reviews[reviews["business_id"].astype(str).isin(restaurant_ids)]
tips_r     = tips[tips["business_id"].astype(str).isin(restaurant_ids)]
checkins_r = checkins[checkins["business_id"].astype(str).isin(restaurant_ids)]

print("\n--- Restaurant-only events ---")
print("reviews  :", len(reviews_r))
print("tips     :", len(tips_r))
print("checkins :", len(checkins_r))

# =========================
# Build Overview page outputs
# =========================

# 1) overview_stats.json  (STRICTLY restaurants)
overview_stats = {
    "total_users": int(users["user_id"].nunique()),
    "reviews": int(len(reviews_r)),          # sample review events
    "restaurants": int(biz_rest["business_id"].nunique()),
    "tips": int(len(tips_r)),
    "checkins": int(len(checkins_r)),
}

with open(os.path.join(OUT_DIR, "overview_stats.json"), "w", encoding="utf-8") as f:
    json.dump(overview_stats, f, indent=2)

print("\n✅ Wrote overview_stats.json")
print(overview_stats)

# =========================
# 2) top_cities.csv
#    Sample review activity per city
# =========================
reviews_geo = reviews_r.merge(
    biz_rest[["business_id","city","state"]],
    on="business_id",
    how="inner"
)

top_cities = (
    reviews_geo
    .dropna(subset=["city","state"])
    .groupby(["city","state"], as_index=False)
    .agg(
        restaurants=("business_id","nunique"),
        sample_reviews=("review_id","count"),
    )
    .sort_values(["restaurants","sample_reviews"], ascending=False)
    .head(10)
)

top_cities.to_csv(os.path.join(OUT_DIR, "top_cities.csv"), index=False)
print("✅ Wrote top_cities.csv")
display(top_cities)

# =========================
# 3) state_coverage.csv
# =========================
state_coverage = (
    biz_rest
    .dropna(subset=["state"])
    .groupby("state", as_index=False)
    .agg(restaurants=("business_id","nunique"))
    .sort_values("restaurants", ascending=False)
)

state_coverage.to_csv(os.path.join(OUT_DIR, "state_coverage.csv"), index=False)
print("✅ Wrote state_coverage.csv")
display(state_coverage.head(10))

print("\n🎉 Done. Drag these into public/data/:")
print(" - overview_stats.json")
print(" - top_cities.csv")
print(" - state_coverage.csv")

✅ Loaded files
users   : (10000, 6)
reviews : (35017, 8)
tips    : (5423, 5)
checkins: (22693, 2)
biz(all): (23597, 10)

🍽️ Restaurant-only businesses: (14027, 10)

--- Uniqueness ---
unique users        : 10000
unique restaurants  : 14027
unique biz in reviews: 22220

--- Restaurant-only events ---
reviews  : 23778
tips     : 3954
checkins : 13990

✅ Wrote overview_stats.json
{'total_users': 10000, 'reviews': 23778, 'restaurants': 14027, 'tips': 3954, 'checkins': 13990}
✅ Wrote top_cities.csv


,city,state,restaurants,sample_reviews
355,Philadelphia,PA,1657,3604
209,Indianapolis,IN,816,1422
309,New Orleans,LA,774,2192
305,Nashville,TN,764,1608
471,Tampa,FL,720,1271
494,Tucson,AZ,688,1127
402,Saint Louis,MO,527,1014
380,Reno,NV,520,1137
408,Santa Barbara,CA,320,801
37,Boise,ID,306,541


✅ Wrote state_coverage.csv


,state,restaurants
13,PA,3312
5,FL,2129
10,MO,1290
14,TN,1216
8,IN,1192
9,LA,1097
1,AZ,821
11,NJ,749
12,NV,665
2,CA,490



🎉 Done. Drag these into public/data/:
 - overview_stats.json
 - top_cities.csv
 - state_coverage.csv
